# 01 — Data Quality (Phase 1)

การตรวจสอบคุณภาพข้อมูล OHLCV ของสัญญา S50 ที่ดึงผ่าน tvkit ในเฟส 1

**ROADMAP §1.5 deliverables** — heatmap ของ missing candles, การกระจายของ return รายปี / รายเซสชัน, การเปลี่ยนแปลงของ volume / open interest ข้าม rollover, และการกระจายของ spread

All code cells are English; markdown cells follow the csm-set convention of Thai narrative.

In [1]:
from __future__ import annotations

import polars as pl

from tfex_s50_multi_tf_swing.config.settings import get_settings
from tfex_s50_multi_tf_swing.data import ParquetStore, SessionCalendar, Validator

settings = get_settings()
store = ParquetStore(settings.data_dir)
validator = Validator(calendar=SessionCalendar(roll_offset_days=settings.roll_offset_days))

## 1. Missing-candle heatmap per session

In [2]:
# Read 5m continuous and bucket gaps by (date, session).
df = store.read_continuous("5m")
df.head()

time,timeframe,open,high,low,close,volume,contract_at_time,adjustment_factor
"datetime[μs, UTC]",str,"decimal[18,4]","decimal[18,4]","decimal[18,4]","decimal[18,4]","decimal[18,4]",str,"decimal[18,8]"
2025-03-24 02:45:00 UTC,"""5m""",719.0000,719.9000,715.5000,715.7000,5660.0000,"""S501!""",1.00000000
2025-03-24 02:50:00 UTC,"""5m""",715.7000,717.2000,714.7000,717.1000,2601.0000,"""S501!""",1.00000000
2025-03-24 02:55:00 UTC,"""5m""",717.1000,717.2000,715.5000,716.5000,1637.0000,"""S501!""",1.00000000
2025-03-24 03:00:00 UTC,"""5m""",716.6000,717.4000,715.3000,715.8000,4447.0000,"""S501!""",1.00000000
2025-03-24 03:05:00 UTC,"""5m""",715.8000,716.0000,714.3000,714.4000,3428.0000,"""S501!""",1.00000000


## 2. Return distribution by year / by session

In [3]:
# Compute log returns and group by year / by session.
returns = df.with_columns(
    (pl.col("close").cast(pl.Float64).log() - pl.col("close").cast(pl.Float64).shift(1).log()).alias("ret")
).drop_nulls("ret")
returns.describe()

statistic,time,timeframe,open,high,low,close,volume,contract_at_time,adjustment_factor,ret
str,str,str,f64,f64,f64,f64,f64,str,f64,f64
"""count""","""20280""","""20280""",20280.0,20280.0,20280.0,20280.0,20280.0,"""20280""",20280.0,20280.0
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,"""0""",0.0,0.0
"""mean""","""2025-10-28 15:50:35.739645+00:…",null,826.857288,827.555754,826.180296,826.866642,2104.200099,null,1.0,0.000018
"""std""",null,null,98.655236,98.67472,98.633335,98.662161,1963.141607,null,0.0,0.001689
"""min""","""2025-03-24 02:50:00+00:00""","""5m""",620.0,622.7,618.8,620.0,1.0,"""S501!""",1.0,-0.079336
"""25%""","""2025-07-14 09:00:00+00:00""",null,753.4,754.2,752.4,753.3,934.0,null,1.0,-0.000611
"""50%""","""2025-10-29 04:00:00+00:00""",null,812.2,812.7,811.7,812.2,1492.0,null,1.0,0.0
"""75%""","""2026-02-12 07:25:00+00:00""",null,920.7,921.7,919.6,920.6,2503.0,null,1.0,0.000617
"""max""","""2026-06-04 09:50:00+00:00""","""5m""",1041.0,1041.3,1039.4,1041.0,29257.0,"""S501!""",1.0,0.057099


## 3. Volume / open-interest evolution across rollovers

In [4]:
df.group_by("contract_at_time").agg(pl.col("volume").sum().alias("total_volume")).sort("contract_at_time")

contract_at_time,total_volume
str,"decimal[38,4]"
"""S501!""",42678838.0000


## 4. Spread distribution

In [5]:
df.with_columns(
    ((pl.col("high").cast(pl.Float64) - pl.col("low").cast(pl.Float64)) / pl.col("close").cast(pl.Float64)).alias("spread_frac")
).select("spread_frac").describe()

statistic,spread_frac
str,f64
"""count""",20281.0
"""null_count""",0.0
"""mean""",0.001683
"""std""",0.001243
"""min""",0.0
"""25%""",0.000953
"""50%""",0.001356
"""75%""",0.002026
"""max""",0.02875
